# Reproducibility Demo

This notebook demonstrates the system's **hard reproducibility requirement**:

> Any published result must be reconstructable from recorded inputs.

We verify this by:
1. Taking a published result from the catalog.
2. Reading its raw files directly from disk via `labdata.store`.
3. Recomputing the sha256 of each raw file.
4. Asserting that the recomputed hashes match the `raw_file_hashes_json`
   recorded in the reproducibility receipt at publish time.

If the assertion passes, the raw files on disk are byte-for-byte identical to
what was hashed and recorded when the result was published — confirming that
the result can be reconstructed.

## Why this matters

The reproducibility receipt (architecture §5) records:
- `raw_file_hashes_json` — sha256 of every raw file
- `manifest_hash` — hash of the ingest manifest
- `parameter_version_ids_json` — parameter versions used for processing
- `parser_version` — exact version of the parser code
- `processing_git_sha` — Git SHA of the processing codebase

Together these ensure that even after system updates, instrument replacements,
or years of lab operation, a published result can be traced back to its
exact input bytes and the exact code that produced it.

In [ ]:
import hashlib
import json

import labdata.catalog as catalog
import labdata.store as store

## 1. Seed demo data (if needed)

If the catalog is empty, seed a demo run that has been driven all the way to
the `published` state. The seeder writes real files to `LABDATA_ROOT` and
inserts a proper reproducibility receipt into the database.

In [ ]:
df_pub = catalog.published_results()

if df_pub.empty:
    print("No published results found — seeding demo data...")
    from labdata.seed import seed_demo
    seeded = seed_demo()
    print(f"Seeded: {seeded}")
    df_pub = catalog.published_results()

print(f"Published results available: {len(df_pub)}")
df_pub[['id', 'run_id', 'parser_version', 'published_by', 'published_at']].head()

## 2. Pick a published result and read its receipt

In [ ]:
pub = df_pub.iloc[0]
pub_result_id = pub['id']
run_id = pub['run_id']

print(f"Published result ID: {pub_result_id}")
print(f"Run ID:              {run_id}")
print(f"Parser version:      {pub.get('parser_version')}")
print(f"Git SHA:             {pub.get('processing_git_sha')}")
print(f"Published by:        {pub.get('published_by')}")
print(f"Published at:        {pub.get('published_at')}")

# Load the raw_file_hashes_json from the receipt
raw_hashes_raw = pub.get('raw_file_hashes_json')
if raw_hashes_raw is None:
    raise RuntimeError("published result has no raw_file_hashes_json — receipt incomplete")

# raw_file_hashes_json is a list of {name, sha256} (the Go publisher's shape).
# psycopg returns jsonb already parsed; tolerate a str or dict shape too.
if isinstance(raw_hashes_raw, str):
    raw_hashes_raw = json.loads(raw_hashes_raw)
if isinstance(raw_hashes_raw, dict):
    receipt_hashes = dict(raw_hashes_raw)
else:
    receipt_hashes = {fh['name']: fh['sha256'] for fh in raw_hashes_raw}
if not receipt_hashes:
    raise RuntimeError('reproducibility receipt has no raw file hashes')
print(f"\nFiles in receipt:")
for fname, sha in receipt_hashes.items():
    print(f"  {fname}: {sha}")

## 3. Re-read raw files from disk and recompute sha256

`store.raw_files(run_id)` returns paths to the actual files on disk.
We read each file and compute its sha256 independently.

In [ ]:
def sha256_file(path) -> str:
    """Compute the sha256 hex digest of a file."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()


# Resolve raw file paths via the catalog (sample_id, device_id -> path)
raw_paths = store.raw_files(run_id)

if not raw_paths:
    raise RuntimeError(f"No raw files found on disk for run {run_id}")

print(f"Files found on disk for run {run_id[:12]}...:")
recomputed_hashes = {}
for p in raw_paths:
    sha = sha256_file(p)
    recomputed_hashes[p.name] = sha
    print(f"  {p.name}: {sha}")

## 4. Assert hashes match the receipt

This is the key verification step: if the hashes match, the raw files on disk
are identical to what was hashed at publish time.

In [ ]:
print("Comparing recomputed hashes against the reproducibility receipt...\n")

failures = []
matches = []

for fname, receipt_sha in receipt_hashes.items():
    disk_sha = recomputed_hashes.get(fname)
    if disk_sha is None:
        failures.append(f"  MISSING on disk: {fname}")
    elif disk_sha != receipt_sha:
        failures.append(
            f"  MISMATCH: {fname}\n"
            f"    receipt:    {receipt_sha}\n"
            f"    recomputed: {disk_sha}"
        )
    else:
        matches.append(f"  OK: {fname}")

# Check for files on disk not in receipt
for fname in recomputed_hashes:
    if fname not in receipt_hashes:
        failures.append(f"  EXTRA (not in receipt): {fname}")

print("Results:")
for m in matches:
    print(m)
for f in failures:
    print(f)

# Hard assertion — this must pass for the result to be considered reproducible
assert not failures, (
    f"Reproducibility check FAILED for published_result_id={pub_result_id}:\n"
    + "\n".join(failures)
)

print(f"\nAll {len(matches)} file(s) verified — result is reproducible.")

## 5. Verify catalog hashes also match (belt and suspenders)

The `v_run_files` view records sha256 values that were verified at promotion
time. We cross-check the on-disk hashes against the catalog's stored values
as well.

In [ ]:
df_files = catalog.run_files(run_id)

catalog_hash_map = {row['name']: row['sha256'] for _, row in df_files.iterrows()}

print("Cross-check: recomputed vs catalog (v_run_files):\n")
catalog_failures = []
for fname, disk_sha in recomputed_hashes.items():
    cat_sha = catalog_hash_map.get(fname)
    if cat_sha is None:
        catalog_failures.append(f"  NOT IN CATALOG: {fname}")
    elif cat_sha != disk_sha:
        catalog_failures.append(
            f"  MISMATCH: {fname}\n"
            f"    catalog:    {cat_sha}\n"
            f"    recomputed: {disk_sha}"
        )
    else:
        print(f"  OK: {fname}")

assert not catalog_failures, (
    "Catalog hash cross-check FAILED:\n" + "\n".join(catalog_failures)
)
print("\nAll catalog hashes match on-disk files.")

## Summary

The two assertions above (receipt vs disk, catalog vs disk) together confirm:

1. The raw files on disk are **byte-for-byte identical** to what was ingested.
2. The receipt hash equals the catalog hash — the two independent records agree.
3. Therefore the published result can be **reconstructed deterministically** by:
   - Reading these raw files (same bytes, verified).
   - Running the recorded `parser_version` on them.
   - Using the recorded `parameter_version_ids_json`.
   - Using the `processing_git_sha` + `processing_environment_id`.

This is the hard requirement of the lab data system.